In [2]:
import pandas as pd

import numpy as np

import re
import pickle
import faiss

import xgboost as xgb

import lightgbm as lgb

from datasets import load_from_disk

from sentence_transformers import SentenceTransformer

from transformers import pipeline

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
wiki = load_from_disk(
    "../datasets/Wikipedia"
)

In [4]:
index = faiss.read_index(
    "../faiss_index/wiki.index"
)

In [5]:
with open(
    "../embeddings/wiki_chunks.pkl",
    "rb"
) as f:

    wiki_chunks = pickle.load(f)

print("Total chunks:", len(wiki_chunks))

Total chunks: 1076104


In [6]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

c:\Users\SujanRam\OneDrive\Documents\Hallucination project\.venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
nli_pipeline = pipeline(
    "text-classification",

    model="MoritzLaurer/deberta-v3-base-mnli-fever-anli",

    device=0
)

In [8]:
xgb_model = xgb.XGBClassifier()

xgb_model.load_model(
    "../models/xgboost_model.json"
)

In [9]:
lgb_model = lgb.Booster(
    model_file="../models/lightgbm_model.txt"
)

In [16]:
def retrieve_evidence(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    # Search large candidate pool
    distances, indices = index.search(
        query_embedding,
        50
    )

    query_words = set(
        re.findall(r'\w+', query.lower())
    )

    scored_passages = []

    for idx in indices[0]:

        passage = wiki_chunks[int(idx)]

        passage_lower = passage.lower()

        passage_words = set(
            re.findall(r'\w+', passage_lower)
        )

        # =================================================
        # SEMANTIC SIMILARITY
        # =================================================

        passage_embedding = embedding_model.encode(
            [passage],
            convert_to_numpy=True
        )

        sim = cosine_similarity(
            query_embedding,
            passage_embedding
        )[0][0]

        # =================================================
        # KEYWORD OVERLAP
        # =================================================

        overlap = len(
            query_words.intersection(
                passage_words
            )
        )

        # =================================================
        # ENTITY MATCH BONUS
        # =================================================

        entity_bonus = 0

        for word in query_words:

            if word in passage_lower:

                entity_bonus += 1

        # =================================================
        # CAPITAL CITY BONUS
        # =================================================

        capital_bonus = 0

        if "capital" in passage_lower:

            capital_bonus += 3

        # =================================================
        # FINAL SCORE
        # =================================================

        final_score = (
            0.5 * sim
            +
            0.3 * overlap
            +
            0.15 * entity_bonus
            +
            0.05 * capital_bonus
        )

        scored_passages.append(
            (final_score, passage)
        )

    # Sort by reranking score
    scored_passages = sorted(
        scored_passages,
        key=lambda x: x[0],
        reverse=True
    )

    top_passages = [
        p[1]
        for p in scored_passages[:top_k]
    ]

    return top_passages

In [17]:
def extract_features(claim):

    passages = retrieve_evidence(
        claim,
        top_k=2
    )

    claim_embedding = embedding_model.encode(
        [claim],
        convert_to_numpy=True
    )

    similarities = []

    entailment_probs = []

    contradiction_probs = []

    neutral_probs = []

    for passage in passages:

        passage_embedding = embedding_model.encode(
            [passage],
            convert_to_numpy=True
        )

        sim = cosine_similarity(
            claim_embedding,
            passage_embedding
        )[0][0]

        similarities.append(sim)

        # NLI
        result = nli_pipeline(
            {
                "text": claim,
                "text_pair": passage
            }
        )

        label = result["label"].lower()

        score = result["score"]

        entailment = 0
        contradiction = 0
        neutral = 0

        if "entail" in label:

            entailment = score

        elif "contrad" in label:

            contradiction = score

        else:

            neutral = score

        entailment_probs.append(entailment)

        contradiction_probs.append(contradiction)

        neutral_probs.append(neutral)

    # =====================================================
    # ORIGINAL FEATURES
    # =====================================================

    max_similarity = max(similarities)

    mean_similarity = np.mean(similarities)

    max_entailment = max(entailment_probs)

    max_contradiction = max(contradiction_probs)

    max_neutral = max(neutral_probs)

    claim_length = len(claim.split())

    entity_count = len(
        re.findall(r'\b[A-Z][a-z]+\b', claim)
    )

    number_count = len(
        re.findall(r'\d+', claim)
    )

    negation_present = int(
        any(
            word in claim.lower()
            for word in [
                "not",
                "no",
                "never",
                "none"
            ]
        )
    )

    # =====================================================
    # ADDITIONAL FEATURES
    # =====================================================

    evidence_count = len(passages)

    avg_word_length = np.mean(
        [len(word) for word in claim.split()]
    )

    punctuation_count = len(
        re.findall(r'[^\w\s]', claim)
    )

    unique_word_ratio = len(
        set(claim.split())
    ) / max(len(claim.split()), 1)

    uppercase_ratio = sum(
        1 for c in claim if c.isupper()
    ) / max(len(claim), 1)

    temporal_expression_present = int(
        bool(
            re.search(
                r'\b(19|20)\d{2}\b',
                claim
            )
        )
    )

    evidence_conflict_score = (
        max_contradiction
    )

    source_reliability_score = (
        max_similarity
    )

    claim_specificity = (
        unique_word_ratio
    )

    features = {

        "max_similarity": max_similarity,

        "mean_similarity": mean_similarity,

        "max_entailment": max_entailment,

        "max_contradiction": max_contradiction,

        "max_neutral": max_neutral,

        "claim_length": claim_length,

        "entity_count": entity_count,

        "number_count": number_count,

        "negation_present": negation_present,

        "evidence_count": evidence_count,

        "avg_word_length": avg_word_length,

        "punctuation_count": punctuation_count,

        "unique_word_ratio": unique_word_ratio,

        "uppercase_ratio": uppercase_ratio,

        "temporal_expression_present":
            temporal_expression_present,

        "evidence_conflict_score":
            evidence_conflict_score,

        "source_reliability_score":
            source_reliability_score,

        "claim_specificity":
            claim_specificity
    }

    return features, passages

In [18]:
def predict_hallucination(claim):

    features, passages = extract_features(
        claim
    )

    feature_df = pd.DataFrame(
        [features]
    )

    # XGBoost probability
    xgb_prob = xgb_model.predict_proba(
        feature_df
    )[0][1]

    # LightGBM probability
    lgb_prob = lgb_model.predict(
        feature_df
    )[0]

    # Ensemble probability
    final_prob = (
        xgb_prob + lgb_prob
    ) / 2

    label = (
        "Hallucinated"
        if final_prob > 0.5
        else "Factual"
    )

    return {

        "claim": claim,

        "prediction": label,

        "hallucination_probability":
            round(final_prob, 4),

        "retrieved_evidence":
            passages,

        "features":
            features
    }

In [19]:
claim = "Paris is the capital of Germany"

result = predict_hallucination(
    claim
)

print(result)

{'claim': 'Paris is the capital of Germany', 'prediction': 'Factual', 'hallucination_probability': 0.2966, 'retrieved_evidence': ['rship with Paris, France. Every Berlin borough also established its own twin towns. For example, the borough of Friedrichshain-Kreuzberg has a partnership with the Israeli city of Kiryat Yam.\n\nCapital city\nBerlin is the capital of the Federal Republic of Germany. The President of Germany, whose func', 'abitants. It is a university city, was the birthplace of Ludwig van Beethoven and was the capital of West Germany from 1949 to 1990. Bonn was the seat of government of reunited Germany from 1990 to 1999.\n\nFounded in the 1st century BC as a Roman settlement in the province Germania Inferior, Bonn is '], 'features': {'max_similarity': 0.585799, 'mean_similarity': 0.5644846, 'max_entailment': 0, 'max_contradiction': 0.9688833951950073, 'max_neutral': 0.8213191628456116, 'claim_length': 6, 'entity_count': 2, 'number_count': 0, 'negation_present': 0, 'evidenc

In [20]:
result = predict_hallucination(
    claim
)

In [21]:
print("\n===== CLAIM =====\n")

print(result["claim"])

print("\n===== PREDICTION =====\n")

print(result["prediction"])

print("\n===== HALLUCINATION PROBABILITY =====\n")

print(result["hallucination_probability"])

print("\n===== RETRIEVED EVIDENCE =====\n")

for i, evidence in enumerate(
    result["retrieved_evidence"]
):

    print(f"\nEvidence {i+1}:\n")

    print(evidence[:500])

print("\n===== FEATURES =====\n")

for k, v in result["features"].items():

    print(k, ":", v)


===== CLAIM =====

Paris is the capital of Germany

===== PREDICTION =====

Factual

===== HALLUCINATION PROBABILITY =====

0.2966

===== RETRIEVED EVIDENCE =====


Evidence 1:

rship with Paris, France. Every Berlin borough also established its own twin towns. For example, the borough of Friedrichshain-Kreuzberg has a partnership with the Israeli city of Kiryat Yam.

Capital city
Berlin is the capital of the Federal Republic of Germany. The President of Germany, whose func

Evidence 2:

abitants. It is a university city, was the birthplace of Ludwig van Beethoven and was the capital of West Germany from 1949 to 1990. Bonn was the seat of government of reunited Germany from 1990 to 1999.

Founded in the 1st century BC as a Roman settlement in the province Germania Inferior, Bonn is 

===== FEATURES =====

max_similarity : 0.585799
mean_similarity : 0.5644846
max_entailment : 0
max_contradiction : 0.9688833951950073
max_neutral : 0.8213191628456116
claim_length : 6
entity_count : 2
num